In [1]:
from pathlib import Path

import pandas as pd

In [2]:
PROJECT_ROOT = Path.cwd().parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "olist"

RAW_DATA_DIR

WindowsPath('c:/Users/chand/OneDrive/Documents/Playground/delivery-intelligence/data/raw/olist')

In [4]:
csv_files = sorted(RAW_DATA_DIR.glob("*.csv"))

print("Number of CSV files:", len(csv_files))

for file in csv_files:
    print(file.name)

Number of CSV files: 9
olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_orders_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


In [5]:
tables = {
    file.stem: pd.read_csv(file)
    for file in csv_files
}

In [6]:
for name, df in tables.items():
    print(name, df.shape)

olist_customers_dataset (99441, 5)
olist_geolocation_dataset (1000163, 5)
olist_order_items_dataset (112650, 7)
olist_order_payments_dataset (103886, 5)
olist_order_reviews_dataset (99224, 7)
olist_orders_dataset (99441, 8)
olist_products_dataset (32951, 9)
olist_sellers_dataset (3095, 4)
product_category_name_translation (71, 2)


In [7]:
audit_summary = []

for name, df in tables.items():
    audit_summary.append({
        "table": name,
        "rows": len(df),
        "columns": df.shape[1],
        "duplicate_rows": df.duplicated().sum(),
        "missing_cells": df.isna().sum().sum()
    })

audit_summary_df = pd.DataFrame(audit_summary)

audit_summary_df

,table,rows,columns,duplicate_rows,missing_cells
0,olist_customers_dataset,99441,5,0,0
1,olist_geolocation_dataset,1000163,5,261831,0
2,olist_order_items_dataset,112650,7,0,0
3,olist_order_payments_dataset,103886,5,0,0
4,olist_order_reviews_dataset,99224,7,0,145903
5,olist_orders_dataset,99441,8,0,4908
6,olist_products_dataset,32951,9,0,2448
7,olist_sellers_dataset,3095,4,0,0
8,product_category_name_translation,71,2,0,0


In [8]:
orders = tables["olist_orders_dataset"]

orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [9]:
orders.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date']

In [10]:
print("Rows:", len(orders))
print("Unique order IDs:", orders["order_id"].nunique())
print("Duplicate order IDs:", orders["order_id"].duplicated().sum())
print("Unique customer IDs:", orders["customer_id"].nunique())

Rows: 99441
Unique order IDs: 99441
Duplicate order IDs: 0
Unique customer IDs: 99441


In [11]:
customers = tables["olist_customers_dataset"]

customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [12]:
customers.columns.tolist()

['customer_id',
 'customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state']

In [13]:
print("Rows:", len(customers))
print("Unique customer IDs:", customers["customer_id"].nunique())
print("Duplicate customer IDs:", customers["customer_id"].duplicated().sum())
print("Unique customer_unique_ids:", customers["customer_unique_id"].nunique())

Rows: 99441
Unique customer IDs: 99441
Duplicate customer IDs: 0
Unique customer_unique_ids: 96096


In [14]:
order_customer_check = orders["customer_id"].isin(customers["customer_id"])

print("Orders with matching customer:", order_customer_check.sum())
print("Orders without matching customer:", (~order_customer_check).sum())

Orders with matching customer: 99441
Orders without matching customer: 0


In [15]:
customer_order_check = customers["customer_id"].isin(orders["customer_id"])

print("Customer records used by an order:", customer_order_check.sum())
print("Customer records without an order:", (~customer_order_check).sum())

Customer records used by an order: 99441
Customer records without an order: 0


In [16]:
order_items = tables["olist_order_items_dataset"]

order_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [17]:
order_items.columns.tolist()

['order_id',
 'order_item_id',
 'product_id',
 'seller_id',
 'shipping_limit_date',
 'price',
 'freight_value']

In [19]:
print("Rows:", len(order_items))
print("Unique order IDs:", order_items["order_id"].nunique())
print("Unique sellers:", order_items["seller_id"].nunique())

print(
    "Duplicate order_id + order_item_id pairs:",
    order_items.duplicated(subset=["order_id", "order_item_id"]).sum()
)

Rows: 112650
Unique order IDs: 98666
Unique sellers: 3095
Duplicate order_id + order_item_id pairs: 0


In [20]:
items_per_order = order_items.groupby("order_id").size()

print("Minimum items per order:", items_per_order.min())
print("Maximum items per order:", items_per_order.max())
print("Orders with more than 1 item:", (items_per_order > 1).sum())

Minimum items per order: 1
Maximum items per order: 21
Orders with more than 1 item: 9803


In [21]:
item_order_match = order_items["order_id"].isin(orders["order_id"])

print("Item rows with matching order:", item_order_match.sum())
print("Item rows without matching order:", (~item_order_match).sum())

Item rows with matching order: 112650
Item rows without matching order: 0


In [22]:
orders_without_items = orders[
    ~orders["order_id"].isin(order_items["order_id"])
]

print("Orders without item records:", len(orders_without_items))

orders_without_items["order_status"].value_counts(dropna=False)

Orders without item records: 775


order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

In [23]:
sellers_per_order = (
    order_items.groupby("order_id")["seller_id"]
    .nunique()
)

In [24]:
print("Minimum sellers per order:", sellers_per_order.min())
print("Maximum sellers per order:", sellers_per_order.max())
print("Orders with more than 1 seller:", (sellers_per_order > 1).sum())

Minimum sellers per order: 1
Maximum sellers per order: 5
Orders with more than 1 seller: 1278


In [25]:
sellers_per_order.value_counts().sort_index()

seller_id
1    97388
2     1219
3       54
4        3
5        2
Name: count, dtype: int64

In [26]:
payments = tables["olist_order_payments_dataset"]
products = tables["olist_products_dataset"]
reviews = tables["olist_order_reviews_dataset"]
sellers = tables["olist_sellers_dataset"]

# PAYMENTS
payment_rows_per_order = payments.groupby("order_id").size()

print("=== PAYMENTS ===")
print("Rows:", len(payments))
print("Unique orders:", payments["order_id"].nunique())
print("Orders with multiple payment rows:", (payment_rows_per_order > 1).sum())
print("Maximum payment rows for one order:", payment_rows_per_order.max())
print("\nPayment types:")
print(payments["payment_type"].value_counts())

# PRODUCTS
print("\n=== PRODUCTS ===")
print("Rows:", len(products))
print("Unique product IDs:", products["product_id"].nunique())
print("Duplicate product IDs:", products["product_id"].duplicated().sum())
print("\nMissing values by column:")
print(products.isna().sum())

# REVIEWS
review_rows_per_order = reviews.groupby("order_id").size()

print("\n=== REVIEWS ===")
print("Rows:", len(reviews))
print("Unique orders reviewed:", reviews["order_id"].nunique())
print("Orders with multiple review rows:", (review_rows_per_order > 1).sum())
print("Maximum reviews for one order:", review_rows_per_order.max())
print("\nMissing values by column:")
print(reviews.isna().sum())

# SELLERS
print("\n=== SELLERS ===")
print("Rows:", len(sellers))
print("Unique seller IDs:", sellers["seller_id"].nunique())
print("Duplicate seller IDs:", sellers["seller_id"].duplicated().sum())

unmatched_sellers = ~order_items["seller_id"].isin(sellers["seller_id"])
print("Order-item rows with unmatched seller:", unmatched_sellers.sum())

=== PAYMENTS ===
Rows: 103886
Unique orders: 99440
Orders with multiple payment rows: 2961
Maximum payment rows for one order: 29

Payment types:
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

=== PRODUCTS ===
Rows: 32951
Unique product IDs: 32951
Duplicate product IDs: 0

Missing values by column:
product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

=== REVIEWS ===
Rows: 99224
Unique orders reviewed: 98673
Orders with multiple review rows: 547
Maximum reviews for one order: 3

Missing values by column:
review_id                      0
order_id                       0
review_score                   0
review_comment_title       

In [27]:
geo = tables["olist_geolocation_dataset"]

print("=== GEOLOCATION ===")
print("Rows:", len(geo))
print("Exact duplicate rows:", geo.duplicated().sum())
print(
    "Unique ZIP prefixes:",
    geo["geolocation_zip_code_prefix"].nunique()
)

rows_per_zip = geo.groupby("geolocation_zip_code_prefix").size()

print("ZIP prefixes with multiple geolocation rows:",
      (rows_per_zip > 1).sum())
print("Maximum rows for one ZIP prefix:", rows_per_zip.max())

=== GEOLOCATION ===
Rows: 1000163
Exact duplicate rows: 261831
Unique ZIP prefixes: 19015
ZIP prefixes with multiple geolocation rows: 17972
Maximum rows for one ZIP prefix: 1146


In [28]:
orders_dt = orders.copy()

timestamp_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in timestamp_cols:
    orders_dt[col] = pd.to_datetime(orders_dt[col], errors="coerce")

print("=== ORDER STATUS ===")
print(orders_dt["order_status"].value_counts(dropna=False))

print("\n=== MISSING TIMESTAMPS ===")
print(orders_dt[timestamp_cols].isna().sum())

print("\n=== DATE RANGE ===")
print(
    "First purchase:",
    orders_dt["order_purchase_timestamp"].min()
)
print(
    "Last purchase:",
    orders_dt["order_purchase_timestamp"].max()
)

print("\n=== LIFECYCLE ORDERING CHECKS ===")

print(
    "Approval before purchase:",
    (
        orders_dt["order_approved_at"]
        < orders_dt["order_purchase_timestamp"]
    ).sum()
)

print(
    "Carrier handoff before approval:",
    (
        orders_dt["order_delivered_carrier_date"]
        < orders_dt["order_approved_at"]
    ).sum()
)

print(
    "Customer delivery before carrier handoff:",
    (
        orders_dt["order_delivered_customer_date"]
        < orders_dt["order_delivered_carrier_date"]
    ).sum()
)

=== ORDER STATUS ===
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

=== MISSING TIMESTAMPS ===
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

=== DATE RANGE ===
First purchase: 2016-09-04 21:15:19
Last purchase: 2018-10-17 17:30:18

=== LIFECYCLE ORDERING CHECKS ===
Approval before purchase: 0
Carrier handoff before approval: 1359
Customer delivery before carrier handoff: 23


In [29]:
delivered = orders_dt[
    orders_dt["order_status"] == "delivered"
].copy()

print("=== DELIVERED ORDERS ===")
print("Delivered orders:", len(delivered))

print(
    "Missing actual delivery timestamp:",
    delivered["order_delivered_customer_date"].isna().sum()
)

print(
    "Missing estimated delivery date:",
    delivered["order_estimated_delivery_date"].isna().sum()
)

# Check how often the estimated delivery timestamp is exactly midnight
midnight_promises = (
    (delivered["order_estimated_delivery_date"].dt.hour == 0)
    & (delivered["order_estimated_delivery_date"].dt.minute == 0)
    & (delivered["order_estimated_delivery_date"].dt.second == 0)
)

print(
    "Estimated delivery timestamps at midnight:",
    midnight_promises.sum()
)

# Calendar-date lateness candidate
eligible = delivered[
    delivered["order_delivered_customer_date"].notna()
    & delivered["order_estimated_delivery_date"].notna()
].copy()

eligible["late_by_calendar_date"] = (
    eligible["order_delivered_customer_date"].dt.date
    >
    eligible["order_estimated_delivery_date"].dt.date
)

print("\nEligible delivered orders:", len(eligible))
print("Late orders:", eligible["late_by_calendar_date"].sum())
print(
    "Late rate:",
    eligible["late_by_calendar_date"].mean()
)

=== DELIVERED ORDERS ===
Delivered orders: 96478
Missing actual delivery timestamp: 8
Missing estimated delivery date: 0
Estimated delivery timestamps at midnight: 96478

Eligible delivered orders: 96470
Late orders: 6534
Late rate: 0.06773090079817559


In [30]:
orders_without_payment = orders[
    ~orders["order_id"].isin(payments["order_id"])
]

orders_without_payment

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
30710,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,2016-10-04 00:00:00


In [31]:
approval_anomalies = orders_dt[
    orders_dt["order_delivered_carrier_date"]
    < orders_dt["order_approved_at"]
]

delivery_anomalies = orders_dt[
    orders_dt["order_delivered_customer_date"]
    < orders_dt["order_delivered_carrier_date"]
]

print("Carrier before approval:", len(approval_anomalies))
print("Delivery before carrier:", len(delivery_anomalies))

display(approval_anomalies.head())
display(delivery_anomalies.head())

Carrier before approval: 1359
Delivery before carrier: 23


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
15,dcb36b511fcac050b97cd5c05de84dc3,3b6828a50ffe546942b7a473d70ac0fc,delivered,2018-06-07 19:03:12,2018-06-12 23:31:02,2018-06-11 14:54:00,2018-06-21 15:34:32,2018-07-04
64,688052146432ef8253587b930b01a06d,81e08b08e5ed4472008030d70327c71f,delivered,2018-04-22 08:48:13,2018-04-24 18:25:22,2018-04-23 19:19:14,2018-04-24 19:31:58,2018-05-15
199,58d4c4747ee059eeeb865b349b41f53a,1755fad7863475346bc6c3773fe055d3,delivered,2018-07-21 12:49:32,2018-07-26 23:31:53,2018-07-24 12:57:00,2018-07-25 23:58:19,2018-07-31
210,412fccb2b44a99b36714bca3fef8ad7b,c6865c523687cb3f235aa599afef1710,delivered,2018-07-22 22:30:05,2018-07-23 12:31:53,2018-07-23 12:24:00,2018-07-24 19:26:42,2018-07-31
415,56a4ac10a4a8f2ba7693523bb439eede,78438ba6ace7d2cb023dbbc81b083562,delivered,2018-07-22 13:04:47,2018-07-27 23:31:09,2018-07-24 14:03:00,2018-07-28 00:05:39,2018-08-06


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
6437,a1abeb653a4d4cd1e142ccb8c82cd069,5f50465da00b7fed5dd1239f4ecf6e2c,delivered,2017-07-20 11:20:52,2017-07-21 06:43:14,2017-07-28 16:57:58,2017-07-25 19:32:56,2017-08-14
9553,383aa8b2724fe452d9ccd9934a8c628b,b1cb2f9d7a19480f3749e248db14d58f,delivered,2017-07-02 20:58:43,2017-07-02 21:10:20,2017-07-07 17:22:41,2017-07-06 14:27:51,2017-07-21
13487,cb1134f9010d242e9515ad1c78ec0c39,2fd33ac77677bd214b1882868317eeed,delivered,2017-07-16 12:35:34,2017-07-18 06:03:50,2017-07-20 19:22:02,2017-07-19 14:13:28,2017-08-08
14474,dceb62e8fa94b46006c9554fed743df0,2721900eb4e0f1cc2c836dd7bc1b1e11,delivered,2017-07-20 20:58:05,2017-07-22 11:45:11,2017-08-01 18:23:30,2017-07-26 18:09:10,2017-08-11
19268,5f9d46795c3126674e52becb3a1a517f,79287bcaafdde5c793b996fc40bb7d9f,delivered,2017-07-18 11:48:20,2017-07-18 12:03:29,2017-07-20 23:03:42,2017-07-20 18:52:41,2017-07-31


In [32]:
orders_dt[
    orders_dt["order_purchase_timestamp"] >= "2018-08-01"
]["order_status"].value_counts()

order_status
delivered      6351
canceled        103
shipped          48
invoiced         23
unavailable       7
Name: count, dtype: int64

In [33]:
REPORTS_DIR = PROJECT_ROOT / "report"
REPORTS_DIR.mkdir(exist_ok=True)

audit_summary_df.to_csv(
    REPORTS_DIR / "table_quality_summary.csv",
    index=False
)

In [35]:
schema_inventory = []

for table_name, df in tables.items():
    for column in df.columns:
        schema_inventory.append({
            "table": table_name,
            "column": column,
            "dtype": str(df[column].dtype),
            "non_null_count": df[column].notna().sum(),
            "missing_count": df[column].isna().sum(),
            "missing_pct": round(df[column].isna().mean() * 100, 2),
            "unique_values": df[column].nunique(dropna=True)
        })

schema_inventory_df = pd.DataFrame(schema_inventory)

schema_inventory_df.to_csv(
    REPORTS_DIR / "schema_inventory.csv",
    index=False
)

schema_inventory_df.head(10)

,table,column,dtype,non_null_count,missing_count,missing_pct,unique_values
0,olist_customers_dataset,customer_id,object,99441,0,0.0,99441
1,olist_customers_dataset,customer_unique_id,object,99441,0,0.0,96096
2,olist_customers_dataset,customer_zip_code_prefix,int64,99441,0,0.0,14994
3,olist_customers_dataset,customer_city,object,99441,0,0.0,4119
4,olist_customers_dataset,customer_state,object,99441,0,0.0,27
5,olist_geolocation_dataset,geolocation_zip_code_prefix,int64,1000163,0,0.0,19015
6,olist_geolocation_dataset,geolocation_lat,float64,1000163,0,0.0,717360
7,olist_geolocation_dataset,geolocation_lng,float64,1000163,0,0.0,717613
8,olist_geolocation_dataset,geolocation_city,object,1000163,0,0.0,8011
9,olist_geolocation_dataset,geolocation_state,object,1000163,0,0.0,27


In [38]:
import hashlib
import json

def sha256_file(path):
    sha256 = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            sha256.update(chunk)

    return sha256.hexdigest()


archive_path = PROJECT_ROOT / "data" / "raw" / "Ecommerce Dataset.zip"

source_files = [archive_path] + sorted(RAW_DATA_DIR.glob("*.csv"))

source_manifest = {
    "dataset": "Brazilian E-Commerce Public Dataset by Olist",
    "source": "Official Kaggle dataset page",
    "acquisition_date": "2026-10-05",
    "license": "CC BY-NC-SA 4.0",
    "files": []
}

for file in source_files:
    source_manifest["files"].append({
        "path": str(file.relative_to(PROJECT_ROOT)).replace("\\", "/"),
        "size_bytes": file.stat().st_size,
        "sha256": sha256_file(file)
    })

with (REPORTS_DIR / "source_manifest.json").open("w", encoding="utf-8") as file:
    json.dump(source_manifest, file, indent=2)

source_manifest

{'dataset': 'Brazilian E-Commerce Public Dataset by Olist',
 'source': 'Official Kaggle dataset page',
 'acquisition_date': '2026-10-05',
 'license': 'CC BY-NC-SA 4.0',
 'files': [{'path': 'data/raw/Ecommerce Dataset.zip',
   'size_bytes': 44717580,
   'sha256': '967e41e04fc306fe604e2a693f488995a8b41e5047418f8a5c8e4abd6deca784'},
  {'path': 'data/raw/olist/olist_customers_dataset.csv',
   'size_bytes': 9033957,
   'sha256': '983a422239e1712ded753b3bf9ecf47dc73f144d306029dcfa99e70a226883d2'},
  {'path': 'data/raw/olist/olist_geolocation_dataset.csv',
   'size_bytes': 61273883,
   'sha256': 'b514f6fc991b9566aeba02aa5d67e2c3630f034b60a0e05aa0d082a3b66d88d6'},
  {'path': 'data/raw/olist/olist_order_items_dataset.csv',
   'size_bytes': 15438671,
   'sha256': '0bc4d068c4fe38cbb01bd90e8746e3c613fe7b4baef75fab7b0e329701c3e279'},
  {'path': 'data/raw/olist/olist_order_payments_dataset.csv',
   'size_bytes': 5777138,
   'sha256': '4f713964f2815dbbaa40b9488268c55aac3627bfce5aa96cf58d1f3616de3cc0'